In [2]:
import pandas as pd
import numpy as np

In [3]:
df=pd.read_csv('water_potability.csv')
df = df.drop_duplicates()
df = df.fillna(df.median(numeric_only=True))

In [4]:
from sklearn.model_selection import StratifiedShuffleSplit

splitter = StratifiedShuffleSplit(n_splits=1, test_size=0.2, random_state=123)

for train_indices, test_indices in splitter.split(df, df["Potability"]):
    train_set = df.iloc[train_indices]
    test_set = df.iloc[test_indices]

print("Training set:", len(train_set))
print("Testing set:", len(test_set))

print("\nTraining set distribution:")
print(train_set["Potability"].value_counts())
print(train_set.head())

print("\nTesting set distribution:")
print(test_set["Potability"].value_counts())
print(test_set.head())

Training set: 2620
Testing set: 656

Training set distribution:
Potability
0    1598
1    1022
Name: count, dtype: int64
            ph    Hardness  ...  Turbidity  Potability
1318  6.724639  223.175415  ...   2.921415           0
1979  7.201702  147.711086  ...   3.883250           1
1239  6.810467  209.735559  ...   4.402340           0
946   5.924540  186.620068  ...   4.224343           0
1032  8.697122  166.452774  ...   4.180568           0

[5 rows x 10 columns]

Testing set distribution:
Potability
0    400
1    256
Name: count, dtype: int64
            ph    Hardness  ...  Turbidity  Potability
3202  8.743959  197.749325  ...   4.118973           1
2474  4.964070  168.283365  ...   4.761701           0
1972  5.892413  139.362253  ...   3.311813           1
1937  6.281904  160.306685  ...   4.457730           1
1639  6.758852  218.153477  ...   3.639972           1

[5 rows x 10 columns]


For this project, I selected nine independent variables (X) that represent different water quality measurements: pH, Hardness, Solids, Chloramines, Sulfate, Conductivity, Organic Carbon, Trihalomethanes, and Turbidity.

The target variable (y) is Potability, which indicates whether water is safe to drink. A value of 1 represents potable water, while 0 represents non-potable water.

I chose these features because they describe the chemical and physical characteristics of water that could influence its potability. I decided to use all nine measurements initially rather than removing features, so I can evaluate how well they work together when predicting water potability.

Since Potability is already a binary categorical variable, there is no need to create additional buckets.

In [5]:
x_train=train_set.drop("Potability", axis=1)
y_train=train_set["Potability"]
x_test=test_set.drop("Potability", axis=1)
y_test=test_set["Potability"]

from sklearn.tree import DecisionTreeClassifier
tree_classifier = DecisionTreeClassifier(random_state=42)
tree_classifier.fit(x_train, y_train)
y_predicted = tree_classifier.predict(x_test)


from sklearn.metrics import confusion_matrix
matrix = confusion_matrix(y_test, y_predicted)
print(matrix)

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
print ("Accuracy is ", accuracy_score(y_test, y_predicted))

# We have to specify how to combine for the multiclassifications
print ("Precision is ", precision_score(y_test, y_predicted, average="weighted"))
print ("Sensitivity is ", recall_score(y_test, y_predicted, average="weighted"))
print ("F1 is ", f1_score(y_test, y_predicted, average="weighted"))

[[272 128]
 [120 136]]
Accuracy is  0.6219512195121951
Precision is  0.6241308053154743
Sensitivity is  0.6219512195121951
F1 is  0.622949949779218


The Decision Tree model achieved an accuracy of 62.20%, precision of 62.41%, sensitivity of 62.20%, and an F1-score of 62.29%.
Based on the confusion matrix, the model correctly predicted 272 non-potable water samples and 136 potable water samples. Overall, the model performed okay, it could perform better.

In [6]:
from sklearn.svm import SVC
svm_classifier = SVC()
svm_classifier.fit(x_train, y_train)

#Evaluating the model
from sklearn.metrics import confusion_matrix
y_pred = svm_classifier.predict(x_test)
matrix= confusion_matrix(y_test, y_pred)
print(matrix)

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
print ("Accuracy is ", accuracy_score(y_test, y_pred))
print("Precision is ", precision_score(y_test, y_pred, average="weighted"))
print ("Sensitivity is ", recall_score(y_test, y_pred, average="weighted"))
print ("F1 is ", f1_score(y_test, y_pred, average="weighted"))
print("Support:")

print(svm_classifier.support_)
print(svm_classifier.support_vectors_)
print(svm_classifier.n_support_)


[[400   0]
 [256   0]]
Accuracy is  0.6097560975609756
Precision is  0.37180249851279
Sensitivity is  0.6097560975609756
F1 is  0.4619364375461936
Support:
[   0    2    3 ... 2613 2614 2616]
[[6.72463934e+00 2.23175415e+02 4.15520197e+04 ... 1.31650346e+01
  6.10587957e+01 2.92141477e+00]
 [6.81046652e+00 2.09735559e+02 3.26023401e+04 ... 1.67367486e+01
  4.23494608e+01 4.40233951e+00]
 [5.92454048e+00 1.86620068e+02 4.43761874e+04 ... 1.27581153e+01
  5.94566909e+01 4.22434251e+00]
 ...
 [7.65754173e+00 2.30820008e+02 3.50562322e+04 ... 1.84813964e+01
  7.22050747e+01 4.79252745e+00]
 [6.27746309e+00 2.25941935e+02 2.28073389e+04 ... 1.32256659e+01
  5.08033529e+01 3.12316616e+00]
 [7.03675210e+00 1.68409525e+02 2.61485379e+04 ... 1.06745825e+01
  5.18769548e+01 3.89332683e+00]]
[1078 1022]


C:\Users\s567783\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


 The SVM model achieved an accuracy of 61%, a precision of 37.18%, a sensitivity of 60.98%, and an F1 score of 46.19%. Looking at the confusion matrix, the model predicted every observation as class 0. It correctly classified 400 observations but failed to identify any of the 256 observations in class 1. The model also had 1078 support vectors for class 0 and 1022 for class 1. Overall, I learned that accuracy alone does not tell us how well a model performs, since the SVM had 61% accuracy but failed to predict any class 1 observations.

The Decision Tree performed slightly better than the SVM model. It achieved an accuracy of 62.2% compared to 61% for SVM, and a precision of 62.41% compared to 37.18%. Based on these results, the Decision Tree was the better performing model for this dataset. 